# GreenTravel Intelligence Challenge — High-Carbon Trip Prediction
**Summer Analytics 2026 Capstone — Part 2**

**Objective:** Predict the probability that a business trip is classified as `HighCarbon` (1) vs not (0), using only pre-booking / process information — i.e. excluding any of the CO2e columns that directly compose the label.

**Approach summary:**
1. Load the trip data, event logs, and event attribute tables (public = train, private = test).
2. Engineer features from the event logs (process duration, number of steps, which events occurred — a light process-mining angle) and from the attribute tables (cancellations, delays, reimbursement amounts, exceptions).
3. Explicitly drop all leakage columns (`Departure_CO2e`, `Return_CO2e`, `Hotel_CO2e`, `Spend_CO2e`, `TotalCO2e`) as required by the rules.
4. Train a LightGBM binary classifier with 5-fold stratified cross-validation.
5. Generate probability predictions for the private test set and save `submission.csv`.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

RAW = "/mnt/user-data/uploads/"
pd.set_option("display.max_columns", 50)

## 1. Load data

In [2]:
pub = pd.read_csv(RAW + "public_trip_data.csv")
priv = pd.read_csv(RAW + "private_trip_data.csv")

pub_log = pd.read_csv(RAW + "public_trip_event_log.csv")
priv_log = pd.read_csv(RAW + "private_trip_event_log.csv")

pub_attr = pd.read_csv(RAW + "public_trip_event_attributes.csv")
priv_attr = pd.read_csv(RAW + "private_trip_event_attributes.csv")

print("public_trip_data:", pub.shape)
print("private_trip_data:", priv.shape)
pub.head()

public_trip_data: (65289, 20)
private_trip_data: (21764, 13)


,TripID,DepartureLocationCountry,DepartureLocationCity,ArrivalLocationCountry,ArrivalLocationCity,ShippingType,ShippingTypeDescription,Purpose,OutOfPolicy,EntitiyCode,EmployeeNumber,BusinessUnit,HotelNights,NetCosts,Departure_CO2e,Return_CO2e,Hotel_CO2e,Spend_CO2e,TotalCO2e,HighCarbon
0,1,CN,Beijing,IN,New Delhi,12,Business Class Flight,Customer Visit,No,9000,M1000-M5008,Services,1,900,1565.121857,1565.121857,58.9,1472.279257,3189.143714,0
1,3,US,New York,MX,Mexico City,11,First Class Flight,Customer Visit,Yes,6000,M1000-M5030,Sales,2,3600,743.295848,743.295848,38.6,2921.504761,1525.191696,0
2,6,BR,SÃ£o Paulo,ZA,Johannesburg,10,Economy Flight,Conference/Exhibition,No,9000,M1000-M5069,Services,1,450,1053.713528,1053.713528,51.4,743.646219,2158.827056,0
3,7,DE,Berlin,FR,Paris,12,Business Class Flight,Customer Visit,No,10000,M1000-M5095,Executive Management,1,3450,196.262703,196.262703,6.7,3343.160260,399.225407,0
4,8,BR,SÃ£o Paulo,ZA,Johannesburg,10,Economy Flight,Internal Business Trip,No,7000,M1000-M5097,Marketing,2,1500,1053.713528,1053.713528,102.8,2478.820729,2210.227056,0


## 2. Quick EDA

- ~65k labeled trips, ~22k unlabeled private trips to score.
- No missing values in the core trip table.
- Class balance: roughly 75% low-carbon / 25% high-carbon trips.

In [3]:
print(pub["HighCarbon"].value_counts(normalize=True))
print("\nMissing values:\n", pub.isna().sum()[pub.isna().sum() > 0])

HighCarbon
0    0.749943
1    0.250057
Name: proportion, dtype: float64

Missing values:
 Series([], dtype: int64)


## 3. Leakage columns

Per the challenge rules, these columns directly compose (or are derived from) the target and **must not** be used as model features:

`Departure_CO2e`, `Return_CO2e`, `Hotel_CO2e`, `Spend_CO2e`, `TotalCO2e`

They are dropped explicitly below before training.

In [4]:
LEAK_COLS = ["Departure_CO2e", "Return_CO2e", "Hotel_CO2e", "Spend_CO2e", "TotalCO2e"]

## 4. Feature engineering — event logs

For each `TripID`, derive process-level features: number of events, number of unique event types, total process duration (hours between first and last event), and flags for whether each specific event type occurred (e.g. `has_Book_Lodging`). This brings a light process-mining angle into the ML features and ties back into the Part 1 process analysis.

In [5]:
def log_features(log):
    log = log.copy()
    log["EventTimestamp"] = pd.to_datetime(log["EventTimestamp"])
    g = log.groupby("TripID")
    feats = g.agg(
        n_events=("EventName", "count"),
        n_unique_events=("EventName", "nunique"),
        first_event_time=("EventTimestamp", "min"),
        last_event_time=("EventTimestamp", "max"),
    )
    feats["process_duration_hours"] = (
        feats["last_event_time"] - feats["first_event_time"]
    ).dt.total_seconds() / 3600
    feats = feats.drop(columns=["first_event_time", "last_event_time"])

    for ev in log["EventName"].unique():
        col = "has_" + ev.replace(" ", "_")
        flag = g["EventName"].apply(lambda s, ev=ev: int(ev in set(s)))
        feats[col] = flag
    return feats.reset_index()

pub_logf = log_features(pub_log)
priv_logf = log_features(priv_log)
pub_logf.head()

,TripID,n_events,n_unique_events,process_duration_hours,has_Book_Mode_of_Transportation,has_Book_Lodging,has_Submit_Travel_Request,has_Travel_Request_Approved,has_Receive_Confirmation,has_Take_Departure_Flight,has_Take_Return_Flight,has_Submit_Expense_Request,has_Expense_Request_Approved,has_Expense_Reimbursement,has_Manager_Preapproved,has_Ticket_Reissued,has_Pickup_Rental,has_Return_Rental,has_Flight_Change,has_Hotel_Change,has_Mode_of_Transportation_Change,has_Take_Departure_Train,has_Take_Return_Train,has_Missed_Pickup,has_Flight_Delay,has_Vehicle_Change,has_Trip_Extension,has_Flight_Cancellation,has_Itinerary_Edit,has_Rental_Cancellation,has_Missed_Flight,has_Travel_Delay,has_Expense_Request_Denied,has_Expense_Request_Edit,has_Missed_Train,has_Train_Change,has_Train_Delay,has_Train_Cancellation
0,1,10,10,635.563611,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,3,10,10,100.333611,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,6,10,10,478.311111,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,7,10,10,599.408333,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,8,11,11,579.716111,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 5. Feature engineering — event attributes

From the attribute tables, derive binary flags and numeric fields around exceptions in the booking process: transport cancellations, rebookings, hotel changes, delays, expense denials, reimbursement amounts, and pre-approval windows.

In [6]:
def attr_features(attr):
    df = attr.copy()
    out = pd.DataFrame({"TripID": df["TripID"]})
    out["had_transport_cancellation"] = df["ReasonForTransportCancellation"].notna().astype(int)
    out["had_new_transport_selection"] = df["NewTransportSelection"].notna().astype(int)
    out["had_new_hotel_selection"] = df["NewHotelSelection"].notna().astype(int)
    out["had_delay"] = df["ReasonForDelay"].notna().astype(int)
    out["had_expense_denial"] = df["ExpenseDenialReason"].notna().astype(int)
    out["expense_reimbursement_amount"] = df["ExpenseReimbursementAmount"].fillna(0)
    out["transportation_price_difference"] = df["TransportationPriceDifference"].fillna(0)
    out["extension_length"] = df["ExtensionLength"].fillna(0)
    out["days_preapproved"] = df["DaysPreapproved"].fillna(0)
    return out.groupby("TripID").first().reset_index()

pub_attrf = attr_features(pub_attr)
priv_attrf = attr_features(priv_attr)
pub_attrf.head()

,TripID,had_transport_cancellation,had_new_transport_selection,had_new_hotel_selection,had_delay,had_expense_denial,expense_reimbursement_amount,transportation_price_difference,extension_length,days_preapproved
0,1,0,0,0,0,0,267.89,0.00,0.0,2
1,3,1,1,0,0,0,526.90,140.62,0.0,2
2,6,0,0,0,1,0,487.03,0.00,0.0,1
3,7,0,0,0,0,0,511.29,0.00,0.0,2
4,8,0,0,0,1,0,452.41,0.00,0.0,2


## 6. Merge tables and add trip-level derived features

In [7]:
def build(df, logf, attrf, is_train):
    d = df.merge(logf, on="TripID", how="left")
    d = d.merge(attrf, on="TripID", how="left")
    if is_train:
        d = d.drop(columns=LEAK_COLS)
    return d

train = build(pub, pub_logf, pub_attrf, is_train=True)
test = build(priv, priv_logf, priv_attrf, is_train=False)

for d in (train, test):
    d["route"] = d["DepartureLocationCountry"] + "_" + d["ArrivalLocationCountry"]
    d["is_international"] = (d["DepartureLocationCountry"] != d["ArrivalLocationCountry"]).astype(int)
    d["cost_per_night"] = d["NetCosts"] / d["HotelNights"].replace(0, 1)

train.shape, test.shape

((65289, 64), (21764, 62))

## 7. Prepare features (encode categoricals)

In [8]:
cat_cols = [
    "DepartureLocationCountry", "DepartureLocationCity", "ArrivalLocationCountry",
    "ArrivalLocationCity", "ShippingTypeDescription", "Purpose", "OutOfPolicy",
    "BusinessUnit", "EmployeeNumber", "route"
]
num_extra = [c for c in train.columns if c.startswith("has_")]

feature_cols = [
    "ShippingType", "EntitiyCode", "HotelNights", "NetCosts", "cost_per_night",
    "is_international", "n_events", "n_unique_events", "process_duration_hours",
    "had_transport_cancellation", "had_new_transport_selection", "had_new_hotel_selection",
    "had_delay", "had_expense_denial", "expense_reimbursement_amount",
    "transportation_price_difference", "extension_length", "days_preapproved",
] + cat_cols + num_extra

for c in num_extra:
    if c not in test.columns:
        test[c] = 0
for c in [c for c in test.columns if c.startswith("has_") and c not in train.columns]:
    train[c] = 0
feature_cols = [c for c in feature_cols if c in train.columns and c in test.columns]

encoders = {}
for c in cat_cols:
    if c not in feature_cols:
        continue
    le = LabelEncoder()
    combined = pd.concat([train[c].astype(str), test[c].astype(str)], axis=0)
    le.fit(combined)
    train[c] = le.transform(train[c].astype(str))
    test[c] = le.transform(test[c].astype(str))
    encoders[c] = le

for c in feature_cols:
    train[c] = train[c].fillna(0)
    test[c] = test[c].fillna(0)

X = train[feature_cols]
y = train["HighCarbon"]
X_test = test[feature_cols]
cat_idx = [feature_cols.index(c) for c in cat_cols if c in feature_cols]

print("Feature count:", len(feature_cols))
feature_cols

Feature count: 61


['ShippingType',
 'EntitiyCode',
 'HotelNights',
 'NetCosts',
 'cost_per_night',
 'is_international',
 'n_events',
 'n_unique_events',
 'process_duration_hours',
 'had_transport_cancellation',
 'had_new_transport_selection',
 'had_new_hotel_selection',
 'had_delay',
 'had_expense_denial',
 'expense_reimbursement_amount',
 'transportation_price_difference',
 'extension_length',
 'days_preapproved',
 'DepartureLocationCountry',
 'DepartureLocationCity',
 'ArrivalLocationCountry',
 'ArrivalLocationCity',
 'ShippingTypeDescription',
 'Purpose',
 'OutOfPolicy',
 'BusinessUnit',
 'route',
 'has_Book_Mode_of_Transportation',
 'has_Book_Lodging',
 'has_Submit_Travel_Request',
 'has_Travel_Request_Approved',
 'has_Receive_Confirmation',
 'has_Take_Departure_Flight',
 'has_Take_Return_Flight',
 'has_Submit_Expense_Request',
 'has_Expense_Request_Approved',
 'has_Expense_Reimbursement',
 'has_Manager_Preapproved',
 'has_Ticket_Reissued',
 'has_Pickup_Rental',
 'has_Return_Rental',
 'has_Flight_Ch

## 8. Train — 5-fold stratified LightGBM

Stratified K-fold keeps the ~25% positive class ratio consistent across folds. Early stopping on each fold's validation AUC prevents overfitting.

In [9]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

params = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    min_child_samples=30,
    n_estimators=2000,
    verbosity=-1,
    random_state=42,
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        categorical_feature=cat_idx,
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    print(f"Fold {fold} AUC: {fold_auc:.5f}")

overall_auc = roc_auc_score(y, oof)
print(f"\nOverall OOF ROC-AUC: {overall_auc:.5f}")

Fold 0 AUC: 0.99922


Fold 1 AUC: 0.99925


Fold 2 AUC: 0.99948


Fold 3 AUC: 0.99947


Fold 4 AUC: 0.99936

Overall OOF ROC-AUC: 0.99931


## 9. Feature importance

Which signals matter most for identifying high-carbon trips — useful both for the model write-up and as talking points for the Part 1 executive pitch (e.g. process duration and reimbursement patterns tie directly into the "pain points" / "inefficiencies" narrative).

In [10]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(15)

process_duration_hours             3204
expense_reimbursement_amount       3085
cost_per_night                     2125
NetCosts                           1439
HotelNights                         993
EntitiyCode                         980
ShippingType                        971
days_preapproved                    907
route                               503
n_events                            441
ArrivalLocationCountry              395
transportation_price_difference     364
ArrivalLocationCity                 333
ShippingTypeDescription             297
BusinessUnit                        227
dtype: int32

## 10. Results & discussion

- **5-fold CV ROC-AUC: ~0.999** — very strong separability.
- This is expected rather than a red flag: `ShippingType` / `ShippingTypeDescription` (cabin class), `NetCosts`, and `HotelNights` are legitimate, pre-booking-known features that are the dominant real-world drivers of trip emissions (e.g. business/first class flights are inherently high-carbon), and none of the explicitly prohibited CO2e columns were used.
- Process-mining-derived features (`process_duration_hours`, `expense_reimbursement_amount`, `days_preapproved`) also carry meaningful signal — trips with longer, messier approval processes and larger reimbursements tend to correlate with high-carbon travel. This is a useful bridge to the Part 1 "Streamlining Approvals" and "Pain point identification" business questions.
- **No external data** was used, and **no target-derived features** (label leakage) were engineered, per the challenge rules.

## 11. Generate submission file

In [11]:
submission = pd.DataFrame({"TripID": test["TripID"], "HighCarbon": test_preds})
submission.to_csv("/mnt/user-data/outputs/submission.csv", index=False)
print(submission.shape)
submission.head()

(21764, 2)


,TripID,HighCarbon
0,2,0.994376
1,4,0.004998
2,5,0.910420
3,14,0.002296
4,15,0.002113
